# Entropy, KL Divergence and Mutual Information

Companion notebook for the blog post [*From Thermodynamics to ML: Entropy, KL and Mutual Info*](https://sesen.ai/blog/entropy-kl-divergence-mutual-information).

Follows Bishop §1.6 (*Pattern Recognition and Machine Learning*, 2006).

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zhubarb/sesen_ai_ml_tutorials/blob/main/notebooks/bayesian/entropy_kl_mutual_information.ipynb)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter
from scipy.stats import norm

plt.rcParams.update({'figure.facecolor': 'white', 'axes.facecolor': 'white', 'font.size': 11})

NAVY, TEAL, AMBER = '#1B2D3D', '#3D9B8F', '#D4A24C'

## 1. Shannon entropy of a Bernoulli coin

The simplest case: a single coin with bias $p$. Entropy peaks at $p = 0.5$ where the outcome is maximally unpredictable.

In [ ]:
p = np.linspace(1e-6, 1 - 1e-6, 400)
H_bits = -(p * np.log2(p) + (1 - p) * np.log2(1 - p))

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(p, H_bits, color=NAVY, linewidth=2.5)
ax.fill_between(p, 0, H_bits, color=TEAL, alpha=0.15)
ax.axvline(0.5, color=AMBER, linestyle='--', linewidth=1.2)
ax.set_xlabel('Bias p (probability of heads)')
ax.set_ylabel('Entropy H(p) [bits]')
ax.set_title('Shannon entropy of a Bernoulli coin')
ax.set_xlim(0, 1); ax.set_ylim(0, 1.1)
plt.show()

## 2. Differential entropy of a Gaussian

For a Gaussian with standard deviation $\sigma$, Bishop gives the closed form (eq. 1.110):

$$H[x] = \tfrac{1}{2}(1 + \ln 2\pi\sigma^2)$$

Note that for $\sigma < 1/\sqrt{2\pi e} \approx 0.242$ it is negative — differential entropy does not count bits.

In [ ]:
sigma = np.linspace(0.1, 3.0, 400)
H_gauss = 0.5 * (1 + np.log(2 * np.pi * sigma ** 2))

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(sigma, H_gauss, color=NAVY, linewidth=2.5)
ax.axhline(0, color='gray', linewidth=0.8)
sigma_zero = 1.0 / np.sqrt(2 * np.pi * np.e)
ax.axvline(sigma_zero, color=AMBER, linestyle='--', linewidth=1.2)
ax.set_xlabel('Standard deviation σ')
ax.set_ylabel('Differential entropy H[x] [nats]')
ax.set_title('Gaussian differential entropy')
plt.show()
print(f'H = 0 at σ = {sigma_zero:.4f}')

## 3. KL divergence: mode-seeking vs mean-seeking

Fit a single Gaussian $q$ to a bimodal mixture $p$, by gradient descent on each KL direction.

- **Forward KL** $\mathrm{KL}(p\|q)$ — zero-avoiding, mean-seeking. $q$ spreads to cover both modes.
- **Reverse KL** $\mathrm{KL}(q\|p)$ — zero-forcing, mode-seeking. $q$ collapses onto one mode.

In [ ]:
xs = np.linspace(-6, 6, 800)
dx = xs[1] - xs[0]
p_pdf = 0.5 * norm.pdf(xs, -2, 0.8) + 0.5 * norm.pdf(xs, 2, 0.8)

def q_pdf(mu, log_sigma):
    return norm.pdf(xs, mu, np.exp(log_sigma))

def kl_p_q(mu, log_sigma):
    q = q_pdf(mu, log_sigma) + 1e-12
    return np.sum(p_pdf * (np.log(p_pdf + 1e-12) - np.log(q))) * dx

def kl_q_p(mu, log_sigma):
    q = q_pdf(mu, log_sigma) + 1e-12
    return np.sum(q * (np.log(q) - np.log(p_pdf + 1e-12))) * dx

def gradient_descent(loss_fn, mu0=0.0, log_sigma0=0.1, lr=0.4, steps=50):
    params = np.array([mu0, log_sigma0])
    traj = [params.copy()]
    eps = 1e-3
    for _ in range(steps):
        grad = np.array([
            (loss_fn(params[0] + eps, params[1]) - loss_fn(params[0] - eps, params[1])) / (2 * eps),
            (loss_fn(params[0], params[1] + eps) - loss_fn(params[0], params[1] - eps)) / (2 * eps),
        ])
        params = params - lr * grad
        params[1] = np.clip(params[1], np.log(0.15), np.log(4.0))
        traj.append(params.copy())
    return np.array(traj)

traj_fwd = gradient_descent(kl_p_q, mu0=0.2, log_sigma0=0.0)
traj_rev = gradient_descent(kl_q_p, mu0=1.5, log_sigma0=0.0)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, traj, title in zip(axes, [traj_fwd, traj_rev], ['Forward KL(p||q): mean-seeking', 'Reverse KL(q||p): mode-seeking']):
    ax.plot(xs, p_pdf, color=NAVY, linewidth=2, label='p (target)')
    mu, ls = traj[-1]
    ax.plot(xs, q_pdf(mu, ls), color=AMBER, linewidth=2.5, label='q (fitted)')
    ax.fill_between(xs, 0, q_pdf(mu, ls), color=AMBER, alpha=0.15)
    ax.set_title(title); ax.set_xlabel('x'); ax.legend()
plt.tight_layout(); plt.show()

### Animate the optimisation

Watch both trajectories simultaneously. In Colab the GIF will display inline.

In [ ]:
from IPython.display import Image

n_frames = 26
idx = np.linspace(0, len(traj_fwd) - 1, n_frames).astype(int)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
for ax, title in zip(axes, ['Forward KL(p||q)', 'Reverse KL(q||p)']):
    ax.plot(xs, p_pdf, color=NAVY, linewidth=2)
    ax.set_ylim(0, 0.45); ax.set_xlim(-6, 6); ax.set_title(title); ax.set_xlabel('x')
line_fwd, = axes[0].plot([], [], color=AMBER, linewidth=2.5)
line_rev, = axes[1].plot([], [], color=AMBER, linewidth=2.5)
txt_fwd = axes[0].text(0.02, 0.95, '', transform=axes[0].transAxes, fontsize=9, va='top')
txt_rev = axes[1].text(0.02, 0.95, '', transform=axes[1].transAxes, fontsize=9, va='top')

def update(i):
    k = idx[i]
    mu_f, ls_f = traj_fwd[k]; mu_r, ls_r = traj_rev[k]
    line_fwd.set_data(xs, q_pdf(mu_f, ls_f))
    line_rev.set_data(xs, q_pdf(mu_r, ls_r))
    txt_fwd.set_text(f'step {k}\nμ={mu_f:.2f}, σ={np.exp(ls_f):.2f}')
    txt_rev.set_text(f'step {k}\nμ={mu_r:.2f}, σ={np.exp(ls_r):.2f}')
    return line_fwd, line_rev, txt_fwd, txt_rev

anim = FuncAnimation(fig, update, frames=n_frames, interval=300, blit=True)
anim.save('kl_asymmetry.gif', writer=PillowWriter(fps=4), dpi=100)
plt.close(fig)
Image('kl_asymmetry.gif')

## 4. KL divergence = negative log-likelihood

When you minimise $\mathrm{KL}(p\|q_\theta)$ using samples from $p$, the $p$-dependent term is a constant and what remains is the negative log-likelihood of $\theta$. Fitting a model by MLE is KL minimisation.

In [ ]:
# Demo: fit a Gaussian to samples by minimising empirical NLL (= KL up to constant)
rng = np.random.default_rng(0)
samples = rng.normal(loc=1.5, scale=0.7, size=500)

def nll(mu, sigma):
    return -np.mean(norm.logpdf(samples, mu, sigma))

mus = np.linspace(-1, 4, 200)
sigmas = np.linspace(0.2, 2.0, 200)
MU, SIG = np.meshgrid(mus, sigmas)
L = np.vectorize(nll)(MU, SIG)

fig, ax = plt.subplots(figsize=(7, 4.5))
c = ax.contourf(MU, SIG, L, levels=30, cmap='viridis')
best_idx = np.unravel_index(np.argmin(L), L.shape)
ax.plot(MU[best_idx], SIG[best_idx], 'o', color=AMBER, markersize=12, label='NLL minimum = MLE')
ax.plot(1.5, 0.7, 'x', color='red', markersize=14, label='true (μ, σ)')
ax.set_xlabel('μ'); ax.set_ylabel('σ')
ax.legend(); plt.colorbar(c, ax=ax, label='NLL')
plt.show()
print(f'MLE: μ={MU[best_idx]:.3f}, σ={SIG[best_idx]:.3f}')

## 5. Mutual information between two Gaussians

For a bivariate Gaussian with correlation $\rho$:

$$I(x, y) = -\tfrac{1}{2} \ln(1 - \rho^2)$$

In [ ]:
rho_grid = np.linspace(0, 0.99, 200)
mi = -0.5 * np.log(1 - rho_grid ** 2)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(rho_grid, mi, color=NAVY, linewidth=2.5)
ax.set_xlabel('correlation |ρ|')
ax.set_ylabel('Mutual information [nats]')
ax.set_title('I(x, y) = -½ ln(1 - ρ²) for bivariate Gaussian')
plt.show()

# Empirical check at ρ = 0.7
rho = 0.7
rng = np.random.default_rng(1)
z = rng.standard_normal((5000, 2))
chol = np.array([[1, 0], [rho, np.sqrt(1 - rho ** 2)]])
xy = z @ chol.T
print(f'Theoretical MI at ρ={rho}: {-0.5 * np.log(1 - rho ** 2):.4f} nats')
print(f'Sample correlation: {np.corrcoef(xy.T)[0, 1]:.4f}')

## 6. Information gain: a toy decision tree

Build a four-cluster XOR-like problem. A single axis-aligned split cannot separate the classes but three greedy splits reduce the weighted leaf entropy to zero. That drop is information gain.

In [ ]:
rng = np.random.default_rng(42)
n_per = 80
class0 = np.vstack([rng.normal(loc=[-2, -2], scale=0.6, size=(n_per, 2)),
                    rng.normal(loc=[2, 2], scale=0.6, size=(n_per, 2))])
class1 = np.vstack([rng.normal(loc=[-2, 2], scale=0.6, size=(n_per, 2)),
                    rng.normal(loc=[2, -2], scale=0.6, size=(n_per, 2))])
X = np.vstack([class0, class1])
y = np.array([0] * len(class0) + [1] * len(class1))

def entropy(labels):
    if len(labels) == 0: return 0.0
    _, counts = np.unique(labels, return_counts=True)
    prob = counts / counts.sum()
    return float(-np.sum(prob * np.log2(prob + 1e-12)))

def best_split(mask):
    Xs, ys = X[mask], y[mask]
    base = entropy(ys)
    best = None
    for feat in (0, 1):
        vals = np.sort(np.unique(Xs[:, feat]))
        for thr in 0.5 * (vals[:-1] + vals[1:]):
            left = Xs[:, feat] <= thr
            total = len(ys)
            w = (left.sum() / total) * entropy(ys[left]) + ((~left).sum() / total) * entropy(ys[~left])
            gain = base - w
            if best is None or gain > best[0]:
                best = (gain, feat, thr)
    return best

masks = [np.ones(len(X), dtype=bool)]
weighted_H = [entropy(y)]
for _ in range(3):
    best_info = None
    best_idx = None
    for i, m in enumerate(masks):
        if m.sum() < 4: continue
        info = best_split(m)
        if info is None: continue
        if best_info is None or info[0] > best_info[0]:
            best_info = info; best_idx = i
    gain, feat, thr = best_info
    m = masks[best_idx]
    left = m & (X[:, feat] <= thr)
    right = m & (X[:, feat] > thr)
    masks[best_idx] = left
    masks.append(right)
    total = sum(mm.sum() for mm in masks)
    wH = sum((mm.sum() / total) * entropy(y[mm]) for mm in masks)
    weighted_H.append(wH)

print('Weighted leaf entropy by split:')
for i, h in enumerate(weighted_H):
    print(f'  after {i} splits: H = {h:.4f} bits')

## Exercises

1. **Jensen-Shannon divergence.** Implement $\mathrm{JSD}(p, q) = \tfrac{1}{2}\mathrm{KL}(p\|m) + \tfrac{1}{2}\mathrm{KL}(q\|m)$ where $m = (p+q)/2$. Verify it is symmetric and bounded in $[0, \ln 2]$ nats.
2. **Forward vs reverse on a skewed target.** Replace the bimodal target with a skewed distribution (e.g. a mixture with unequal weights) and see how the two KL directions differ.
3. **Mutual information for a non-Gaussian joint.** Compute MI empirically (via histograms or k-NN) for a pair $(x, y)$ where $y = x^2 + \epsilon$. Correlation is near zero but MI is large.
4. **Conditional entropy.** Verify numerically that $H[x, y] = H[x] + H[y \mid x]$ (Bishop eq. 1.112).

## References

- Bishop, C. M. (2006). *Pattern Recognition and Machine Learning*, Chapter 1.6.
- Cover, T. M. and Thomas, J. A. (2006). *Elements of Information Theory* (2nd ed.).
- MacKay, D. J. C. (2003). *Information Theory, Inference, and Learning Algorithms*. Free PDF: <https://www.inference.org.uk/itila/>.
